In [1]:
pip install google-play-scraper

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install app-store-scraper


  Attempting uninstall: chardet

    Found existing installation: chardet 5.2.0

   ---------------------------------------- 0/5 [chardet]
    Uninstalling chardet-5.2.0:
   ---------------------------------------- 0/5 [chardet]
      Successfully uninstalled chardet-5.2.0
   ---------------------------------------- 0/5 [chardet]
   ---------------------------------------- 0/5 [chardet]
   ---------------------------------------- 0/5 [chardet]
   ---------------------------------------- 0/5 [chardet]
   ---------------------------------------- 0/5 [chardet]
   ---------------------------------------- 0/5 [chardet]
   ---------------------------------------- 0/5 [chardet]
   ---------------------------------------- 0/5 [chardet]
   ---------------------------------------- 0/5 [chardet]
   ---------------------------------------- 0/5 [chardet]
   ---------------------------------------- 0/5 [chardet]
  Attempting uninstall: urllib3
   ---------------------------------------- 0/5 [charde

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
anaconda-client 1.14.0 requires urllib3>=1.26.4, but you have urllib3 1.25.11 which is incompatible.
conda 25.11.1 requires requests<3,>=2.28.0, but you have requests 2.23.0 which is incompatible.
conda-libmamba-solver 25.11.0 requires requests<3,>=2.28.0, but you have requests 2.23.0 which is incompatible.
conda-repo-cli 1.0.173 requires requests>=2.31.0, but you have requests 2.23.0 which is incompatible.
conda-repo-cli 1.0.173 requires urllib3>=2.2.2, but you have urllib3 1.25.11 which is incompatible.
distributed 2025.11.0 requires urllib3>=1.26.5, but you have urllib3 1.25.11 which is incompatible.
jupyterlab-server 2.28.0 requires requests>=2.31, but you have requests 2.23.0 which is incompatible.
pygithub 2.8.1 requires urllib3>=1.26.0, but you have urllib3 1.25.11 which is incompatible.
selenium 4.40.0 req

In [4]:
!pip install pandas tqdm

In [5]:
# =========================================================
# OTT 앱 리뷰 통합 수집기
# - Google Play + Apple App Store
# - Coupang Play / Netflix / TVING / Disney+
# - 기간 필터링 가능
# - CSV 저장
# =========================================================

# 설치 필요 패키지
# pip install google-play-scraper app-store-scraper pandas tqdm

import pandas as pd
from tqdm import tqdm
from datetime import datetime

# -----------------------------
# Google Play
# -----------------------------
from google_play_scraper import reviews, Sort

# -----------------------------
# Apple App Store
# -----------------------------
from app_store_scraper import AppStore


# =========================================================
# 수집 기간 설정
# =========================================================

START_DATE = datetime(2025, 4, 1)
END_DATE   = datetime(2026, 4, 30)


# =========================================================
# 앱 정보
# =========================================================

APPS = {
    "Coupang Play": {
        "google_play_id": "com.coupang.mobile.play",
        "appstore_id": 1558720674
    },
    "Netflix": {
        "google_play_id": "com.netflix.mediaclient",
        "appstore_id": 363590051
    },
    "TVING": {
        "google_play_id": "net.cj.cjhv.gs.tving",
        "appstore_id": 1481335069
    },
    "Disney+": {
        "google_play_id": "com.disney.disneyplus",
        "appstore_id": 1446075923
    }
}


# =========================================================
# Google Play 리뷰 수집
# =========================================================

def collect_google_reviews(app_name, app_id, count=5000):

    print(f"\n[Google Play] {app_name} 수집 시작")

    result, _ = reviews(
        app_id,
        lang='ko',
        country='kr',
        sort=Sort.NEWEST,
        count=count
    )

    rows = []

    for r in tqdm(result):

        review_date = r['at']

        if START_DATE <= review_date <= END_DATE:

            rows.append({
                "app": app_name,
                "store": "Google Play",
                "date": review_date,
                "rating": r['score'],
                "review": r['content'],
                "thumbs_up": r['thumbsUpCount'],
                "version": r.get('reviewCreatedVersion'),
            })

    return pd.DataFrame(rows)


# =========================================================
# App Store 리뷰 수집
# =========================================================

def collect_appstore_reviews(app_name, app_id):

    print(f"\n[App Store] {app_name} 수집 시작")

    app = AppStore(
        country="kr",
        app_name=app_name,
        app_id=app_id
    )

    app.review()

    rows = []

    for r in tqdm(app.reviews):

        review_date = r['date']

        if START_DATE <= review_date <= END_DATE:

            rows.append({
                "app": app_name,
                "store": "App Store",
                "date": review_date,
                "rating": r['rating'],
                "review": r['review'],
                "thumbs_up": None,
                "version": r.get('version')
            })

    return pd.DataFrame(rows)


# =========================================================
# 전체 수집
# =========================================================

all_reviews = []

for app_name, info in APPS.items():

    # -------------------------
    # Google Play
    # -------------------------
    try:
        gp_df = collect_google_reviews(
            app_name,
            info["google_play_id"],
            count=5000
        )

        all_reviews.append(gp_df)

    except Exception as e:
        print(f"[ERROR][Google Play] {app_name}: {e}")

    # -------------------------
    # App Store
    # -------------------------
    try:
        ios_df = collect_appstore_reviews(
            app_name,
            info["appstore_id"]
        )

        all_reviews.append(ios_df)

    except Exception as e:
        print(f"[ERROR][App Store] {app_name}: {e}")


# =========================================================
# 데이터 통합
# =========================================================

final_df = pd.concat(all_reviews, ignore_index=True)

# 중복 제거
final_df.drop_duplicates(
    subset=["app", "store", "date", "review"],
    inplace=True
)

# 날짜 정렬
final_df = final_df.sort_values("date", ascending=False)

# =========================================================
# CSV 저장
# =========================================================

filename = "ott_reviews_2025_2026.csv"

final_df.to_csv(
    filename,
    index=False,
    encoding="utf-8-sig"
)

print("\n====================================")
print("수집 완료")
print(f"총 리뷰 수: {len(final_df):,}")
print(f"저장 파일: {filename}")
print("====================================")


# =========================================================
# 간단 확인
# =========================================================

print(final_df.head())

ModuleNotFoundError: No module named 'urllib3.packages.six.moves'

In [6]:
pip install urllib3==1.26.18

  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.25.11
    Uninstalling urllib3-1.25.11:
      Successfully uninstalled urllib3-1.25.11
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.173 requires requests>=2.31.0, but you have requests 2.23.0 which is incompatible.
conda-repo-cli 1.0.173 requires urllib3>=2.2.2, but you have urllib3 1.26.18 which is incompatible.
requests 2.23.0 requires urllib3!=1.25.0,!=1.25.1,<1.26,>=1.21.1, but you have urllib3 1.26.18 which is incompatible.
selenium 4.40.0 requires urllib3[socks]<3.0,>=2.6.3, but you have urllib3 1.26.18 which is incompatible.
sphinx 8.2.3 requires requests>=2.30.0, but you have requests 2.23.0 which is incompatible.
streamlit 1.51.0 requires requests<3,>=2.27, but you have requests 2.23.0 which is incompatible.


In [7]:
!pip install requests==2.31.0
!pip install six

  Attempting uninstall: requests
    Found existing installation: requests 2.23.0
    Uninstalling requests-2.23.0:
      Successfully uninstalled requests-2.23.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
app-store-scraper 0.3.5 requires requests==2.23.0, but you have requests 2.31.0 which is incompatible.
conda-repo-cli 1.0.173 requires urllib3>=2.2.2, but you have urllib3 1.26.18 which is incompatible.


In [8]:
!pip uninstall app-store-scraper -y

Found existing installation: app-store-scraper 0.3.5
Uninstalling app-store-scraper-0.3.5:
  Successfully uninstalled app-store-scraper-0.3.5


In [9]:
!pip install app-store-web-scraper


  Attempting uninstall: urllib3

    Found existing installation: urllib3 1.26.18

   ---------------------------------------- 0/2 [urllib3]
    Uninstalling urllib3-1.26.18:
   ---------------------------------------- 0/2 [urllib3]
      Successfully uninstalled urllib3-1.26.18
   ---------------------------------------- 0/2 [urllib3]
   ---------------------------------------- 0/2 [urllib3]
   ---------------------------------------- 0/2 [urllib3]
   ---------------------------------------- 0/2 [urllib3]
   ---------------------------------------- 0/2 [urllib3]
   ---------------------------------------- 0/2 [urllib3]
   -------------------- ------------------- 1/2 [app-store-web-scraper]
   ---------------------------------------- 2/2 [app-store-web-scraper]

